# Qwen3-8B CLASH control fine-tuning

This notebook fine-tunes `Qwen/Qwen3-8B` on one of two 98-example non-ecological CLASH control arms: `prompt_only` or `action`. The prompt-only arm preserves the completed user-only causal objective. The action arm keeps the identical CLASH dilemma as one `user` message, adds the exact source `action` field as one assistant response, masks the complete user and generation prefix, and applies loss only to the action text and terminating EOS token. Neither arm includes CLASH's acceptable or unacceptable rationale, character descriptions, or a preferred-action label.

Both arms use the same pinned Qwen revision, BF16 LoRA configuration, optimizer settings, 98 examples, three epochs, effective batch size, seed, and 1,024-token no-truncation rule as the ecological dilemma runs. The dataset text is never prefixed with a literal `User:` string: Qwen's chat template supplies all role-control tokens with thinking disabled. The completed run, tokenizer, adapter, epoch checkpoints, exact chats, token statistics, metrics, environment metadata, and hashes are first written locally and then copied to Google Drive, freshly remounted, and rehashed.

Evaluation retains the same ecological `extreme_v2` primary suite and supervision-matched readout battery used by the previous dilemma notebook. The separate six-template non-ecological control-evaluation suite is omitted. Both verified evaluation bundles are saved beneath the source Drive run and published under the CLASH experiment's isolated GitHub result root.

In [ ]:
from pathlib import Path
import importlib
import os
import subprocess
import sys

REPO_URL = "https://github.com/shengweiming/value-misalignment.git"
REPO_DIR = Path("/content/value-misalignment")

if not (REPO_DIR / ".git").exists():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", "main", REPO_URL, str(REPO_DIR)],
        check=True,
    )
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    check=True,
)
# Colab's optional TorchAO build can conflict with PEFT. This workflow uses
# ordinary BF16 LoRA and does not use TorchAO quantization.
subprocess.run(
    [sys.executable, "-m", "pip", "uninstall", "-y", "torchao"],
    check=True,
)
REPOSITORY_COMMIT = subprocess.run(
    ["git", "rev-parse", "HEAD"],
    cwd=REPO_DIR,
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
# A warm Colab runtime may still hold modules imported from an older checkout.
# Remove only this repository's package modules after the pull so later cells
# import the code at REPOSITORY_COMMIT rather than stale in-memory objects.
stale_project_modules = [
    module_name
    for module_name in tuple(sys.modules)
    if module_name == "scripts" or module_name.startswith("scripts.")
]
for module_name in stale_project_modules:
    del sys.modules[module_name]
importlib.invalidate_caches()
required_export_path = REPO_DIR / "scripts/ecological_prompt_sft/__init__.py"
assert "CLASH_TRAINING_ARMS" in required_export_path.read_text(), (
    "The checked-out repository does not contain the CLASH action arm. "
    f"Checkout: {REPOSITORY_COMMIT}"
)
print("Repository commit:", REPOSITORY_COMMIT)

In [ ]:
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

from google.colab import drive
drive.mount("/content/drive")

import torch

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > GPU, then retry."
assert torch.cuda.is_bf16_supported(), "This workflow requires a BF16-capable GPU."
gpu = torch.cuda.get_device_properties(0)
gpu_memory_gib = gpu.total_memory / 2**30
assert gpu_memory_gib >= 38, "Select an A100 40 GB (or larger) runtime."
print(f"GPU: {gpu.name} ({gpu_memory_gib:.1f} GiB)")

## Configuration

Set `TRAINING_ARM` to `prompt_only` or `action`. Both arms use Qwen3-8B at the same immutable revision, BF16 LoRA over all linear layers, rank 16, alpha 32, dropout 0.05, exactly three epochs, learning rate `1e-4`, micro-batch size 1, gradient accumulation 16, maximum length 1,024, and seed 42. Arm-specific pair names and output roots prevent checkpoint or result reuse across objectives.

In [ ]:
from google.colab import userdata
from scripts.ecological_prompt_sft import CLASH_TRAINING_ARMS, DilemmaSFTConfig

TRAINING_ARM = "action"  # prompt_only | action
assert TRAINING_ARM in CLASH_TRAINING_ARMS
DATASET_PATHS = {
    "prompt_only": Path("data/control_dilemmas/clash/v1/records.jsonl"),
    "action": Path("data/control_dilemmas/clash/sft/action/records.jsonl"),
}
EXPECTED_RECORDS_SHA256 = {
    "prompt_only": "5e8baa5b3043768c466cb667c4211708a86be3257ab11fbfe288a1f9ac8856a4",
    "action": "c094c4329978dcab7c754c38b9dd6a4b934b456d54124e24ceb70b38881b4378",
}
PAIR_NAMES = {
    "prompt_only": "qwen3_8b_clash_prompt_control_sft",
    "action": "qwen3_8b_clash_action_sft",
}
OUTPUT_SLUGS = {
    "prompt_only": "clash_prompt_control_qwen3_8b",
    "action": "clash_action_qwen3_8b",
}
LOCAL_OUTPUT_ROOTS = {
    arm: Path("/content/value-misalignment-runs") / OUTPUT_SLUGS[arm]
    for arm in CLASH_TRAINING_ARMS
}
DRIVE_OUTPUT_ROOTS = {
    arm: Path("/content/drive/MyDrive/value-misalignment") / OUTPUT_SLUGS[arm]
    for arm in CLASH_TRAINING_ARMS
}
FORCE_RETRAIN = False
FORCE_EVALUATION = False
PUBLISH_TO_GITHUB = True
GITHUB_REPOSITORY = "shengweiming/value-misalignment"
GITHUB_BRANCH = "main"

def config_for_arm(arm):
    return DilemmaSFTConfig(
        output_root=LOCAL_OUTPUT_ROOTS[arm],
        training_arm=arm,
        dataset_path=DATASET_PATHS[arm],
        pair_name=PAIR_NAMES[arm],
        base_model="Qwen/Qwen3-8B",
        model_revision="b968826d9c46dd6066d109eabc6255188de91218",
        max_length=1024,
        num_train_epochs=3,
        learning_rate=1e-4,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=16,
        lora_rank=16,
        lora_alpha=32,
        lora_dropout=0.05,
        eval_batch_size=4,
        seed=42,
        cost_counts=(0, 1, 10, 100, 1_000, 10_000, 100_000, 1_000_000),
    )

CONFIGS = {arm: config_for_arm(arm) for arm in CLASH_TRAINING_ARMS}
CONFIG = CONFIGS[TRAINING_ARM]

GITHUB_TOKEN = None
if PUBLISH_TO_GITHUB:
    try:
        GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
    except Exception as exc:
        raise RuntimeError(
            "Add a Colab secret named GITHUB_TOKEN and grant this notebook access before training."
        ) from exc
    if not GITHUB_TOKEN:
        raise RuntimeError(
            "GITHUB_TOKEN is empty; publication is enabled and required by this workflow."
        )

print("Training arm:", CONFIG.training_arm)
print("Pair name:", CONFIG.pair_name)
print("Dataset:", CONFIG.dataset_path)
print("N values:", CONFIG.cost_counts)
CONFIGS

## Dataset and exact loss audit

This cell checks the actual Qwen rendering and the collated training batch before model weights are loaded. The notebook never inserts a literal `User:` prefix. In `prompt_only`, every rendered non-padding user-chat token is supervised. In `action`, the Qwen user and assistant-generation prefix is masked to `-100`; only the exact CLASH action and terminating EOS token remain supervised. The same label-preserving collator used by the corrected ecological and human response arms is tested here end to end.

In [ ]:
from IPython.display import Markdown, display
import pandas as pd
from transformers import AutoTokenizer
from scripts.ecological_prompt_sft import (
    IGNORE_INDEX,
    PromptOnlyCollator,
    load_training_examples,
    tokenize_training_examples,
)

examples, source_manifest = load_training_examples(
    CONFIG.dataset_path,
    training_arm=CONFIG.training_arm,
)
assert len(examples) == 98
assert source_manifest["training_arm"] == TRAINING_ARM
assert source_manifest["records_sha256"] == EXPECTED_RECORDS_SHA256[TRAINING_ARM]
assert source_manifest["contains_normative_labels"] is False
assert source_manifest["contains_assistant_responses"] is (TRAINING_ARM == "action")
if TRAINING_ARM == "action":
    assert source_manifest["assistant_target_field"] == "action"
    assert source_manifest["contains_rationales"] is False

preview_tokenizer = AutoTokenizer.from_pretrained(
    CONFIG.base_model,
    revision=CONFIG.model_revision,
    use_fast=True,
)
if preview_tokenizer.pad_token_id is None:
    assert preview_tokenizer.eos_token_id is not None
    preview_tokenizer.pad_token = preview_tokenizer.eos_token
preview_tokenizer.padding_side = "right"

tokenized_examples = tokenize_training_examples(
    preview_tokenizer,
    examples,
    training_arm=TRAINING_ARM,
    max_length=CONFIG.max_length,
)
assert len(tokenized_examples) == 98
first_messages = [{"role": "user", "content": examples[0]["dilemma"]}]

if TRAINING_ARM == "prompt_only":
    assert all(row["labels"] == row["input_ids"] for row in tokenized_examples)
    assert all(
        row["supervised_token_count"] == row["sequence_length"]
        for row in tokenized_examples
    )
    first_prefix = preview_tokenizer.apply_chat_template(
        first_messages,
        tokenize=False,
        add_generation_prompt=False,
        enable_thinking=False,
    )
    first_full_ids = preview_tokenizer.encode(first_prefix, add_special_tokens=False)
    assert first_full_ids == tokenized_examples[0]["input_ids"]
    assert first_full_ids == tokenized_examples[0]["labels"]
    first_answer = None
else:
    first_answer = examples[0]["assistant_answer"]
    first_prefix = preview_tokenizer.apply_chat_template(
        first_messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )
    first_full = first_prefix + first_answer + preview_tokenizer.eos_token
    first_prefix_ids = preview_tokenizer.encode(first_prefix, add_special_tokens=False)
    first_full_ids = preview_tokenizer.encode(first_full, add_special_tokens=False)
    first_row = tokenized_examples[0]
    assert first_full_ids == first_row["input_ids"]
    assert first_full_ids[:len(first_prefix_ids)] == first_prefix_ids
    assert first_row["labels"][:len(first_prefix_ids)] == [IGNORE_INDEX] * len(first_prefix_ids)
    assert first_row["labels"][len(first_prefix_ids):] == first_full_ids[len(first_prefix_ids):]
    assert all(
        row["supervised_token_count"] < row["sequence_length"]
        for row in tokenized_examples
    )

audit_features = [tokenized_examples[0], tokenized_examples[1]]
audit_batch = PromptOnlyCollator(preview_tokenizer.pad_token_id)(audit_features)
for index, feature in enumerate(audit_features):
    width = len(feature["labels"])
    assert audit_batch["labels"][index, :width].tolist() == feature["labels"]
    assert torch.all(audit_batch["labels"][index, width:] == IGNORE_INDEX)
if TRAINING_ARM == "action":
    assert torch.all(
        audit_batch["labels"][0, :len(first_prefix_ids)] == IGNORE_INDEX
    )

answer_block = (
    "\n\n**Assistant (the exact CLASH focal action; not a preferred-action label):** "
    f"{first_answer}"
) if first_answer is not None else ""
display(Markdown(
    f"### First training example — `{examples[0]['id']}`\n\n"
    "**User** (no literal `User:` prefix is added to the dataset text)\n\n"
    f"{examples[0]['dilemma']}"
    f"{answer_block}"
))
sequence_lengths = pd.Series([row["sequence_length"] for row in tokenized_examples])
supervised_lengths = pd.Series([row["supervised_token_count"] for row in tokenized_examples])
display(pd.DataFrame([{
    "arm": TRAINING_ARM,
    "examples": len(tokenized_examples),
    "minimum_sequence_tokens": int(sequence_lengths.min()),
    "mean_sequence_tokens": float(sequence_lengths.mean()),
    "maximum_sequence_tokens": int(sequence_lengths.max()),
    "minimum_supervised_tokens": int(supervised_lengths.min()),
    "median_supervised_tokens": float(supervised_lengths.median()),
    "mean_supervised_tokens": float(supervised_lengths.mean()),
    "maximum_supervised_tokens": int(supervised_lengths.max()),
    "assistant_turns": int(TRAINING_ARM == "action"),
}]))
print("Rendered Qwen prefix:", repr(first_prefix[:160]))
print("Exact loss check passed through tokenization and collation for:", TRAINING_ARM)

In [ ]:
from scripts.ecological_prompt_sft import (
    find_compatible_complete_run,
    persist_run_to_colab_drive,
    run_dilemma_sft,
    validate_complete_run,
)

artifacts = None
if not FORCE_RETRAIN:
    artifacts = find_compatible_complete_run(DRIVE_OUTPUT_ROOTS[TRAINING_ARM], CONFIG)

if artifacts is None:
    print("No compatible completed CLASH checkpoint found; starting Qwen fine-tuning.")
    local_artifacts = run_dilemma_sft(CONFIG)
    artifacts = persist_run_to_colab_drive(
        local_artifacts,
        DRIVE_OUTPUT_ROOTS[TRAINING_ARM],
    )
    print("TRAINING COMPLETE — copied, freshly remounted, and hash-verified on Drive.")
else:
    print("TRAINING SKIPPED — reusing a compatible hash-verified Drive run.")

validate_complete_run(artifacts)
print("Verified source run:", artifacts.run_dir)

In [ ]:
import json

complete = json.loads(artifacts.complete_marker_path.read_text())
train_metrics = json.loads(artifacts.train_metrics_path.read_text())
dataset_manifest = json.loads(artifacts.dataset_manifest_path.read_text())
run_metadata = json.loads(artifacts.metadata_path.read_text())

assert complete["status"] == "complete"
assert run_metadata["status"] == "complete"
EXPECTED_OBJECTIVES = {
    "prompt_only": "prompt_only_causal_lm",
    "action": "clash_action_response_only_sft_v1",
}
assert run_metadata["training_arm"] == TRAINING_ARM
assert run_metadata["training_objective"] == EXPECTED_OBJECTIVES[TRAINING_ARM]
assert run_metadata["pair_name"] == PAIR_NAMES[TRAINING_ARM]
assert run_metadata["dataset"]["records_sha256"] == EXPECTED_RECORDS_SHA256[TRAINING_ARM]
assert dataset_manifest["example_count"] == 98
assert dataset_manifest["training_arm"] == TRAINING_ARM
assert dataset_manifest["contains_normative_labels"] is False
assert dataset_manifest["contains_assistant_responses"] is (TRAINING_ARM == "action")
assert train_metrics["training_arm"] == TRAINING_ARM
assert train_metrics["training_objective"] == EXPECTED_OBJECTIVES[TRAINING_ARM]

summary = {
    "run": artifacts.run_dir.name,
    "pair_name": run_metadata["pair_name"],
    "adapter_revision": run_metadata["resolved_revisions"]["final_adapter_sha256"],
    "training_examples": train_metrics["training_example_count"],
    "sequence_token_mean": dataset_manifest["tokenization"]["sequence_tokens"]["mean"],
    "supervised_token_mean": dataset_manifest["tokenization"]["supervised_tokens"]["mean"],
    "maximum_tokens": dataset_manifest["tokenization"]["sequence_tokens"]["max"],
}
display(pd.DataFrame([summary]))
print("Saved chat template:", dataset_manifest["chat_template"])

## Primary ecological evaluation

This is the same eight-template, eight-cost `extreme_v2` evaluation used for the ecological dilemma interventions. It scores the unchanged base checkpoint and the CLASH adapter with exact `Yes`/`No` sequence probabilities and thinking disabled. All 64 prompts are displayed before inference.

In [ ]:
from scripts.ecological_prompt_sft import EXTREME_V2_TEMPLATES, build_extreme_v2_cases

primary_preview_cases = build_extreme_v2_cases(CONFIG.cost_counts)
assert len(primary_preview_cases) == len(EXTREME_V2_TEMPLATES) * len(CONFIG.cost_counts)
print(f"Reviewing all {len(primary_preview_cases)} primary cases before inference.")
for case in primary_preview_cases:
    display(Markdown(f"### `{case['case_id']}`\n\n{case['prompt']}"))

In [ ]:
from scripts.ecological_prompt_sft import run_extreme_v2_workflow

primary_workflow = run_extreme_v2_workflow(
    artifacts,
    cost_counts=CONFIG.cost_counts,
    batch_size=CONFIG.eval_batch_size,
    force_evaluation=FORCE_EVALUATION,
)
if primary_workflow.evaluation_reused:
    print("PRIMARY EVALUATION SKIPPED — reusing verified Drive results.")
else:
    print("PRIMARY EVALUATION COMPLETE — verified after a fresh Drive remount.")
primary_workflow.validation

In [ ]:
from IPython.display import Image

primary_eval = primary_workflow.evaluation_artifacts
primary_scores = pd.read_csv(primary_eval.raw_scores_path)
assert len(primary_scores) == 2 * len(primary_preview_cases)
assert set(primary_scores["model_role"]) == {"base", "aligned"}
display(primary_scores.pivot(
    index=["template", "cost_count"],
    columns="model_role",
    values=["p_implement", "semantic_logit_implement"],
).sort_index())
display(pd.read_csv(primary_eval.thresholds_path))
display(Image(filename=str(primary_eval.plot_path)))
print("Verified primary Drive bundle:", primary_eval.output_dir)

## Supervision-matched readout battery

Every readout retains the original eight `extreme_v2` scenario bodies and all eight cost values. Reversed `Yes`/`No` asks about the human-protective action, so `No` is mapped back to the ecological option. Counterbalanced `A`/`B` presents both option orders. Complete option-text scoring also presents both orders and scores each entire option; its primary margin uses mean log-probability per candidate token. In every case, positive `semantic_logit_implement` and larger `p_implement` mean greater support for the ecological option.

In [ ]:
from scripts.ecological_prompt_sft import build_supervision_matched_readout_cases

readout_preview_cases = build_supervision_matched_readout_cases(CONFIG.cost_counts)
assert len(readout_preview_cases) == 320
assert len({case["case_id"] for case in readout_preview_cases}) == 320
display(pd.DataFrame(readout_preview_cases).groupby(
    ["readout_type", "readout_variant"]
).size().rename("case_count").reset_index())
print("Reviewing one cost-1 prompt for each of the 40 readout templates.")
for case in readout_preview_cases:
    if case["cost_count"] == 1:
        display(Markdown(
            f"### `{case['case_id']}`\n\n"
            f"Ecological candidate: `{case['candidate_implement']}`  \n"
            f"Human candidate: `{case['candidate_reject']}`  \n\n{case['prompt']}"
        ))

In [ ]:
from scripts.ecological_prompt_sft import run_supervision_matched_readout_workflow

readout_workflow = run_supervision_matched_readout_workflow(
    artifacts,
    cost_counts=CONFIG.cost_counts,
    batch_size=CONFIG.eval_batch_size,
    force_evaluation=FORCE_EVALUATION,
)
if readout_workflow.evaluation_reused:
    print("READOUT EVALUATION SKIPPED — reusing verified Drive results.")
else:
    print("READOUT EVALUATION COMPLETE — verified after a fresh Drive remount.")
readout_workflow.validation

In [ ]:
readout_eval = readout_workflow.evaluation_artifacts
readout_scores = pd.read_csv(readout_eval.raw_scores_path)
assert len(readout_scores) == 640
assert set(readout_scores["model_role"]) == {"base", "aligned"}
readout_paired = readout_scores.pivot(
    index=["case_id", "readout_type", "readout_variant"],
    columns="model_role",
    values="semantic_logit_implement",
).reset_index()
readout_paired["aligned_minus_base"] = readout_paired["aligned"] - readout_paired["base"]
readout_summary = readout_paired.groupby(
    ["readout_type", "readout_variant"]
)[["base", "aligned", "aligned_minus_base"]].mean().reset_index()
display(readout_summary)
display(pd.read_csv(readout_eval.thresholds_path))
display(Image(filename=str(readout_eval.plot_path)))
print("Verified readout Drive bundle:", readout_eval.output_dir)

In [ ]:
from scripts.ecological_prompt_sft import publish_results_to_github

if PUBLISH_TO_GITHUB:
    publications = {}
    for label, evaluation in {
        "primary_extreme_v2": primary_eval,
        "supervision_matched_readouts": readout_eval,
    }.items():
        publication = publish_results_to_github(
            evaluation,
            source_run_name=artifacts.run_dir.name,
            github_repository=GITHUB_REPOSITORY,
            branch=GITHUB_BRANCH,
            github_token=GITHUB_TOKEN,
            repo_root=REPO_DIR,
        )
        publications[label] = publication
        print(f"{label} GitHub publication verified: {publication.html_url}")
else:
    print("GitHub publication disabled; both verified Drive bundles remain intact.")